# Mission 2 — Annotation Thématique des Avis Clients

**Objectif :** Extraire automatiquement les grandes thématiques présentes dans les avis clients grâce à la **Factorisation en Matrices Non-Négatives (NMF)**, puis mesurer leur importance avec une **Forêt Aléatoire**.

**Méthodologie :**
1. Charger les configurations vectorisées (uniquement celles sans stopwords)
2. Tester les hyperparamètres NMF (Frobenius / Kullback-Leibler, 4 / 5 thèmes)
3. Évaluer la qualité des thèmes via un Random Forest
4. Identifier et annoter les thématiques du meilleur modèle
5. Analyser l'importance de chaque thème

---
## 1. Imports et chargement des données

In [ ]:
import os
import glob
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.decomposition import NMF
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print("Bibliothèques chargées.")

In [ ]:
# On ne garde QUE les fichiers SANS stopwords (S1) : 
# les thèmes sont plus lisibles sans mots vides
chemin_pkl = 'vectorisation-du-texte/output/'
fichiers_s1 = sorted(glob.glob(os.path.join(chemin_pkl, '*_S1_*_FINAL.pkl')))

print(f"{len(fichiers_s1)} configurations sans stopwords trouvées :")
for f in fichiers_s1:
    print(f"  • {os.path.basename(f)}")

---
## 2. Recherche des meilleurs hyperparamètres NMF

On teste **2 fonctions de perte** (Frobenius, Kullback-Leibler) × **2 nombres de thèmes** (4, 5) sur chaque configuration de texte. La qualité des thèmes extraits est évaluée indirectement : on utilise la matrice des poids thématiques (W) comme features pour un **Random Forest** de classification positif/négatif. Un bon score signifie que les thèmes capturent bien la structure des avis.

In [ ]:
# Grille d'hyperparamètres NMF demandée par le sujet
parametres_nmf = [
    {'n_components': 4, 'beta_loss': 'frobenius',       'solver': 'cd'},
    {'n_components': 5, 'beta_loss': 'frobenius',       'solver': 'cd'},
    {'n_components': 4, 'beta_loss': 'kullback-leibler', 'solver': 'mu'},
    {'n_components': 5, 'beta_loss': 'kullback-leibler', 'solver': 'mu'},
]

resultats = []

for chemin_fichier in fichiers_s1:
    nom_config = os.path.basename(chemin_fichier).replace('_FINAL.pkl', '')
    
    with open(chemin_fichier, 'rb') as f:
        data = pickle.load(f)
    X, y = data['X_normalized'], data['target']
    
    for params in parametres_nmf:
        try:
            # Extraction des thèmes
            nmf = NMF(
                n_components=params['n_components'],
                beta_loss=params['beta_loss'],
                solver=params['solver'],
                init='nndsvda',
                random_state=42,
                max_iter=500
            )
            W = nmf.fit_transform(X)
            
            # Évaluation : Random Forest sur les thèmes
            W_train, W_test, y_train, y_test = train_test_split(
                W, y, test_size=0.2, random_state=42
            )
            rf = RandomForestClassifier(n_estimators=100, random_state=42)
            rf.fit(W_train, y_train)
            score = rf.score(W_test, y_test)
            
            resultats.append({
                'Configuration': nom_config,
                'Nb Thèmes': params['n_components'],
                'Fonction de perte': params['beta_loss'],
                'Accuracy RF (%)': round(score * 100, 2)
            })
        except Exception:
            continue

print(f"{len(resultats)} combinaisons évaluées avec succès.")

In [ ]:
# Classement des résultats
df_resultats = (pd.DataFrame(resultats)
                .sort_values('Accuracy RF (%)', ascending=False)
                .reset_index(drop=True))
df_resultats.index += 1

print("TOP 10 — Meilleures combinaisons (NMF + Random Forest)")
print("=" * 65)
display(df_resultats.head(10))

best = df_resultats.iloc[0]
print(f"\n★ Meilleure configuration : {best['Configuration']}")
print(f"  → {best['Nb Thèmes']} thèmes, perte {best['Fonction de perte']}, score {best['Accuracy RF (%)']}%")

---
## 3. Entraînement du meilleur modèle NMF

On ré-entraîne la NMF gagnante pour extraire les thèmes et les analyser en détail.

In [ ]:
# Chargement des données de la config gagnante
chemin_gagnant = os.path.join(chemin_pkl, best['Configuration'] + '_FINAL.pkl')
with open(chemin_gagnant, 'rb') as f:
    data_best = pickle.load(f)

X_best = data_best['X_normalized']
y_best = data_best['target']
vocabulaire = data_best['feature_names']

# Entraînement de la NMF gagnante
solver_best = 'mu' if best['Fonction de perte'] == 'kullback-leibler' else 'cd'

nmf_best = NMF(
    n_components=int(best['Nb Thèmes']),
    beta_loss=best['Fonction de perte'],
    solver=solver_best,
    init='nndsvda',
    random_state=42,
    max_iter=500
)

W_best = nmf_best.fit_transform(X_best)  # Matrice documents × thèmes
H_best = nmf_best.components_             # Matrice thèmes × mots

print(f"Matrice W : {W_best.shape[0]} avis × {W_best.shape[1]} thèmes")
print(f"Matrice H : {H_best.shape[0]} thèmes × {H_best.shape[1]} mots")

---
## 4. Découverte et annotation manuelle des thèmes

On affiche les mots les plus représentatifs de chaque thème, puis on leur attribue un **label humain**.

In [ ]:
# Affichage des 15 mots-clés par thème
n_top = 15
print("MOTS-CLÉS PAR THÈME")
print("=" * 70)
for i, composante in enumerate(H_best):
    top_idx = composante.argsort()[:-n_top - 1:-1]
    top_mots = [vocabulaire[j] for j in top_idx]
    print(f"\nThème {i+1} : {', '.join(top_mots)}")
print("\n" + "=" * 70)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# ANNOTATION MANUELLE — À ADAPTER SELON VOS RÉSULTATS
# ════════════════════════════════════════════════════════════════════
# Observez les mots-clés ci-dessus et attribuez un nom parlant.
# Exemples basés sur les résultats typiques :

labels_themes = {
    1: "Taille / Dimensions du produit",
    2: "Problèmes de livraison / Réception",
    3: "Rapport Qualité-Prix",
    4: "Boucles d'oreilles / Esthétique",
    5: "Satisfaction générale / Cadeau",
}

# Vérification : on affiche mots-clés + label côte à côte
print("THÈMES ANNOTÉS")
print("=" * 70)
for i, composante in enumerate(H_best):
    top_idx = composante.argsort()[:-8 - 1:-1]
    top_mots = ', '.join([vocabulaire[j] for j in top_idx])
    label = labels_themes.get(i + 1, '???')
    print(f"\nThème {i+1} — {label}")
    print(f"  Mots-clés : {top_mots}")
print("\n" + "=" * 70)

---
## 5. Importance des thèmes (Random Forest)

On entraîne un Random Forest sur la matrice W (avis × thèmes) pour prédire le sentiment. L'attribut `feature_importances_` nous indique quels thèmes sont les plus déterminants pour distinguer les avis positifs des négatifs.

In [ ]:
# Entraînement du Random Forest final
W_train, W_test, y_train, y_test = train_test_split(
    W_best, y_best, test_size=0.2, random_state=42
)

rf_final = RandomForestClassifier(n_estimators=200, random_state=42)
rf_final.fit(W_train, y_train)

y_pred = rf_final.predict(W_test)
print("Performance du Random Forest sur les thèmes")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred))

In [ ]:
# Importance de chaque thème
importances = rf_final.feature_importances_
noms_themes = [labels_themes.get(i+1, f'Thème {i+1}') for i in range(len(importances))]

df_importance = (pd.DataFrame({'Thème': noms_themes, 'Importance (%)': importances * 100})
                 .sort_values('Importance (%)', ascending=True))

# Graphique
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(df_importance['Thème'], df_importance['Importance (%)'], color='#2196F3', edgecolor='white')
ax.set_xlabel('Importance (%)', fontsize=12)
ax.set_title('Importance des thématiques pour la classification des avis', fontsize=13, fontweight='bold')

for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.5, bar.get_y() + bar.get_height()/2, f'{width:.1f}%',
            va='center', fontsize=11)

plt.tight_layout()
plt.show()

# Tableau récapitulatif
print("\nRécapitulatif :")
display(df_importance.sort_values('Importance (%)', ascending=False).reset_index(drop=True))

---
## 6. Visualisation : distribution des thèmes dans les avis

Pour chaque avis, le thème dominant est celui avec le poids le plus élevé dans W.

In [ ]:
# Thème dominant par avis
themes_dominants = np.argmax(W_best, axis=1)

df_distrib = pd.DataFrame({
    'Thème dominant': [labels_themes.get(t+1, f'Thème {t+1}') for t in themes_dominants],
    'Sentiment': y_best
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution globale
ordre = [labels_themes[i+1] for i in range(int(best['Nb Thèmes']))]
df_distrib['Thème dominant'].value_counts().reindex(ordre).plot(
    kind='bar', ax=axes[0], color='#2196F3', edgecolor='white'
)
axes[0].set_title('Répartition des avis par thème dominant', fontweight='bold')
axes[0].set_ylabel('Nombre d\'avis')
axes[0].tick_params(axis='x', rotation=30)

# Distribution par sentiment
ct = pd.crosstab(df_distrib['Thème dominant'], df_distrib['Sentiment'])
ct = ct.reindex(ordre)
ct.plot(kind='bar', ax=axes[1], color=['#EF5350', '#66BB6A'], edgecolor='white')
axes[1].set_title('Thème dominant × Sentiment', fontweight='bold')
axes[1].set_ylabel('Nombre d\'avis')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Sentiment')

plt.tight_layout()
plt.show()

---
## Conclusion

La NMF permet d'identifier des thématiques cohérentes dans les avis clients. Le Random Forest confirme que ces thèmes portent une information discriminante vis-à-vis du sentiment. Les thèmes liés aux **problèmes (livraison, taille)** sont typiquement associés aux avis négatifs, tandis que ceux liés à la **satisfaction et aux cadeaux** sont plutôt positifs.